# 01: Setting Up Neo4j Database

This notebook sets up the connection to Neo4j and configures the environment for graph-powered RAG with NotePlan notes.

## Overview

This notebook sets up your Neo4j database connection and prepares it for storing your NotePlan knowledge graph. Think of it as the foundation that all other notebooks build upon.

We'll:
1. Import necessary libraries
2. Configure Neo4j connection using repository settings
3. Test the connection
4. Explore the graph schema
5. **Create vector index** for note embeddings (required for notebook 04)


In [1]:
# Run common imports and setup
%run 00-import.ipynb

# Additional imports specific to this notebook
from graphdatascience import GraphDataScience
from neo4j import GraphDatabase

print("✅ Additional libraries imported")


✅ Added to path: /Users/omareid/Workspace/git/knowledge-agents/src
   Project root: /Users/omareid/Workspace/git/knowledge-agents
✅ Standard library imports loaded
✅ Repository components imported
🔍 Runtime detection: local
✅ Settings loaded:
   Neo4j URI: bolt://localhost:7687
   Neo4j Database: knowledge
   Neo4j Username: neo4j
   Neo4j Password: ********
   LiteLLM Proxy Host: localhost

💡 To override settings, see Settings class docstring:
   help(Settings)  # or help(get_settings)
   # Quick examples:
   # settings = get_settings(neo4j_password='your_password')
   # settings = get_settings(runtime_env='container')

📁 NotePlan directory: /Users/omareid/Library/Containers/co.noteplan.NotePlan3/Data/Library/Application Support/co.noteplan.NotePlan3
   Directory exists: True

✅ Successfully connected to Neo4j
✅ Data manipulation libraries imported
✅ Additional libraries imported


## Connect to Neo4j

We'll use the Graph Data Science Python Client to connect to Neo4j. This client makes it convenient to:
- Display results in a readable format
- Run Graph Data Science algorithms from Python
- Work with vector indexes


In [ ]:
# dependencies, driver, and neo4j_manager are already initialized in 00-import.ipynb
# They are available as: dependencies, driver, neo4j_manager

# Create GDS client for graph data science operations
# Note: For Neo4j Desktop, AURA_DS should be False
# For AuraDS, set to True
AURA_DS = False  # Set to True if using AuraDS

gds = GraphDataScience(
    settings.neo4j_uri,
    auth=(settings.neo4j_username, settings.neo4j_password),
    aura_ds=AURA_DS
)

# Set database (if using multi-database setup)
gds.set_database(settings.neo4j_database)

print("✅ Graph Data Science client initialized")

✅ Graph Data Science client initialized


## Test Connection

Verify the connection works by checking the GDS version.


In [3]:
# Test connection
try:
    version = gds.version()
    print(f"✅ Graph Data Science version: {version}")
except Exception as e:
    print(f"❌ Error connecting to Neo4j: {e}")
    print("Note: If you don't have GDS installed, you can still use the Neo4j driver directly")


✅ Graph Data Science version: 2.23.0


## Explore Graph Schema

View the current graph structure to understand what nodes and relationships exist.


In [4]:
# Get node labels
with driver.session(database=settings.neo4j_database) as session:
    result = session.run("""
        CALL db.labels()
        YIELD label
        RETURN label
        ORDER BY label
    """)
    labels = [record["label"] for record in result]
    print("Node Labels:")
    for label in labels:
        print(f"  - {label}")

# Get relationship types
with driver.session(database=settings.neo4j_database) as session:
    result = session.run("""
        CALL db.relationshipTypes()
        YIELD relationshipType
        RETURN relationshipType
        ORDER BY relationshipType
    """)
    rel_types = [record["relationshipType"] for record in result]
    print("\nRelationship Types:")
    for rel_type in rel_types:
        print(f"  - {rel_type}")


Node Labels:
  - Entity
  - Note

Relationship Types:
  - BELONGS_TO
  - CONTAINS
  - MENTIONS
  - OCCURS_AT
  - REFERENCES
  - RELATED_TO


## Create Vector Index

Create the vector index for note embeddings to enable efficient similarity search.
This index is required before loading vector embeddings in notebook 04.


In [6]:
# Create vector index for note embeddings
# This index enables efficient similarity search using vector embeddings
vector_size = settings.get_embedding_size(settings.litellm_proxy_embedding_model)
neo4j_manager.ensure_vector_index(
    index_name=settings.neo4j_vector_index_name,
    vector_size=vector_size,
    node_label="Note",
    node_property="embedding"
)
print(f"✅ Vector index '{settings.neo4j_vector_index_name}' created/verified")
print(f"   Dimension: {vector_size}")
print(f"   Node label: Note")
print(f"   Property: embedding")

# Verify the index exists
with driver.session(database=settings.neo4j_database) as session:
    result = session.run("""
        SHOW INDEXES
        WHERE name = $index_name
    """, index_name=settings.neo4j_vector_index_name)
    
    index_info = result.single()
    if index_info:
        print(f"   State: {index_info.get('state', 'N/A')}")

# Count nodes
with driver.session(database=settings.neo4j_database) as session:
    result = session.run("""
        MATCH (n)
        RETURN labels(n)[0] as label, COUNT(n) as count
        ORDER BY count DESC
    """)
    print("\nNode Counts:")
    for record in result:
        print(f"  {record['label']}: {record['count']} nodes")


✅ Vector index 'note_embeddings' created/verified
   Dimension: 4096
   Node label: Note
   Property: embedding
   State: ONLINE

Node Counts:
  Note: 193 nodes
  Entity: 134 nodes


## Next Steps

Now that the database is set up with graph schema and vector index, proceed to:
- [**02-extracting-embeddings.ipynb**](./02-extracting-embeddings.ipynb): Generate vector embeddings from NotePlan notes
- [**02o1-extracting-data.ipynb**](./02o1-extracting-data.ipynb): Extract entities and relationships from NotePlan notes
- [**03-loading-data.ipynb**](./03-loading-data.ipynb): Load entities and relationships into Neo4j graph
- [**04o1-loading-vector-embeddings-neo4j.ipynb**](./04o1-loading-vector-embeddings-neo4j.ipynb): Store vector embeddings in Neo4j (requires vector index created above)
